# 14 — CNN 3D sur les images (RQ1, version deep learning)

**Pour Membre A** — necessite un **GPU** (fonctionne aussi sur CPU, mais lentement).

C'est la **branche 3D-CNN** de l'architecture de reference de l'enonce
(Rahim et al., 2022 : 3D-CNN + BRNN). Membre B construit la branche BRNN
de son cote (notebook 12).

**Question :** un vrai reseau de neurones sur les volumes fait-il mieux que les
5 statistiques resumees utilisees au notebook 10 ?

**Quatre configurations comparees, sur les memes plis :**

| Configuration | Entree |
|---|---|
| CNN — IRM seule | volume IRM 64³ |
| CNN — PET seule | volume PET 64³ |
| CNN — IRM + PET | deux volumes, fusion des representations |
| CNN — IRM + PET + tabulaire | + variables cliniques |

> ⚠️ **Seulement 112 participants.** Le surapprentissage est quasi certain :
> on utilise donc dropout, augmentation de donnees et un reseau volontairement
> petit. Si le CNN fait moins bien que le modele tabulaire, **c'est un resultat
> valable** a presenter tel quel.

In [ ]:
# GPU : installer la version CUDA de PyTorch. Sans GPU, retirer l'index-url.
# PyTorch GPU est deja installe dans le .venv du projet (torch+cu118).
# Ne pas relancer pip ici : cela remplacerait la version GPU par la version CPU.
# %pip install torch --quiet

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from IPython.display import display

sys.path.append(r"..\configs")
from config_projet import C

APPAREIL = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Appareil utilise :", APPAREIL)
if APPAREIL.type == "cuda":
    print("GPU :", torch.cuda.get_device_name(0))

# Dossiers produits par les notebooks 07 et 09.
prep = sorted(C.COHORTE.glob("preparation_modele_*"))[-1]
feat = sorted(C.COHORTE.glob("features_image_*"))[-1]
VOLUMES = feat / "volumes_64"
print("Tabulaire :", prep.name)
print("Volumes   :", VOLUMES)

jeu = pd.read_csv(prep / "jeu_modelisation_359.csv")
jeu["PTID"] = jeu["PTID"].astype(str).str.strip().str.upper()

images = pd.read_csv(feat / "features_image.csv")
images["PTID"] = images["PTID"].astype(str).str.strip().str.upper()

donnees = (jeu.merge(images[["PTID"]], on="PTID", how="inner")
              .dropna(subset=["progression"])
              .reset_index(drop=True))
print("Participants avec images et cible :", len(donnees))

In [ ]:
# --- Charger les volumes en memoire (112 x 2 x 64^3 ~ 235 Mo) ---
def charger(modalite):
    piles = []
    for ptid in donnees["PTID"]:
        chemin = VOLUMES / f"{ptid}_{modalite}.npy"
        if not chemin.is_file():
            raise FileNotFoundError(f"Volume manquant : {chemin}")
        piles.append(np.load(chemin).astype(np.float32))
    # Forme finale : (N, 1 canal, 64, 64, 64)
    return np.stack(piles)[:, None, :, :, :]

IRM = charger("IRM")
PET = charger("PET")
y = donnees["progression"].astype(int).to_numpy()

print("IRM :", IRM.shape, "| PET :", PET.shape)
print("Cible : ", len(y), "participants |", int(y.sum()),
      f"progressent ({y.mean() * 100:.1f}%)")

# Axe gauche-droite de chaque volume, dans le tenseur (N, 1, D0, D1, D2) :
# dims 2, 3, 4 = axes numpy 0, 1, 2. Verifie sur les fichiers NIfTI :
#   IRM : axe numpy 0 = gauche-droite  -> dim 2   (le dernier axe est avant-arriere !)
#   PET : axe numpy 2 = gauche-droite  -> dim 4
AXE_GD = {"IRM": 2, "PET": 4}

# --- Variables tabulaires (pour la 4e configuration) ---
TAB_CANDIDATES = [
    "sexe", "education", "age_reference", "APOE_e4",
    "CDR_global", "CDR_SB", "MMSE_debut", "ADAS13", "FAQ_debut",
    "ABETA42", "TAU", "PTAU", "dx_debut",
]
TAB = [c for c in TAB_CANDIDATES if donnees[c].notna().sum() > 0]
TABULAIRE = donnees[TAB].to_numpy(dtype=float)
print("Variables tabulaires :", TAB)

In [ ]:
class Encodeur3D(nn.Module):
    """Petit CNN 3D : 4 blocs conv puis moyenne globale."""
    def __init__(self, canaux=(8, 16, 32, 64)):
        super().__init__()
        couches, precedent = [], 1
        for c in canaux:
            couches += [nn.Conv3d(precedent, c, 3, padding=1),
                        nn.BatchNorm3d(c),
                        nn.ReLU(inplace=True),
                        nn.MaxPool3d(2)]
            precedent = c
        couches.append(nn.AdaptiveAvgPool3d(1))
        self.corps = nn.Sequential(*couches)
        self.dimension = precedent

    def forward(self, x):
        return self.corps(x).flatten(1)


class ReseauFusion(nn.Module):
    """Un encodeur par modalite, puis fusion (+ variables tabulaires)."""
    def __init__(self, n_modalites, n_tabulaire=0):
        super().__init__()
        self.encodeurs = nn.ModuleList([Encodeur3D() for _ in range(n_modalites)])
        dimension = sum(e.dimension for e in self.encodeurs) + n_tabulaire
        self.tete = nn.Sequential(
            nn.Dropout(0.5), nn.Linear(dimension, 32),
            nn.ReLU(inplace=True), nn.Dropout(0.3), nn.Linear(32, 1))

    def forward(self, volumes, tabulaire=None):
        representations = [e(v) for e, v in zip(self.encodeurs, volumes)]
        if tabulaire is not None:
            representations.append(tabulaire)
        return self.tete(torch.cat(representations, dim=1)).squeeze(1)


def augmenter(volumes, axes):
    """Augmentation legere : retournement gauche-droite (sur l'axe propre a chaque
    modalite) + bruit."""
    if torch.rand(1).item() < 0.5:
        volumes = [torch.flip(v, dims=[a]) for v, a in zip(volumes, axes)]
    return [v + 0.05 * torch.randn_like(v) for v in volumes]

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
EPOQUES, LOT = 40, 8

def entrainer_evaluer(listes_volumes, axes, tabulaire, app, test, graine=0):
    """Entraine sur le pli d'apprentissage, renvoie les probabilites du test."""
    torch.manual_seed(graine)

    # Les volumes sont deja normalises (notebook 09).
    V_app = [torch.tensor(v[app], device=APPAREIL) for v in listes_volumes]
    V_test = [torch.tensor(v[test], device=APPAREIL) for v in listes_volumes]

    if tabulaire is None:
        T_app = T_test = None
        n_tab = 0
    else:
        # Imputation (mediane d'apprentissage) puis standardisation.
        brut_app = tabulaire[app]
        mediane = np.nanmedian(brut_app, axis=0)
        mediane = np.where(np.isnan(mediane), 0.0, mediane)
        remplir = lambda X: np.where(np.isnan(X), mediane, X)
        A, T = remplir(brut_app), remplir(tabulaire[test])
        moyenne, ecart = A.mean(axis=0), A.std(axis=0) + 1e-6
        T_app = torch.tensor(((A - moyenne) / ecart), dtype=torch.float32, device=APPAREIL)
        T_test = torch.tensor(((T - moyenne) / ecart), dtype=torch.float32, device=APPAREIL)
        n_tab = A.shape[1]

    y_app = torch.tensor(y[app], dtype=torch.float32, device=APPAREIL)

    modele = ReseauFusion(len(listes_volumes), n_tab).to(APPAREIL)
    positifs = max(int(y[app].sum()), 1)
    poids = torch.tensor([(len(app) - positifs) / positifs],
                         dtype=torch.float32, device=APPAREIL)
    perte = nn.BCEWithLogitsLoss(pos_weight=poids)
    optimiseur = torch.optim.Adam(modele.parameters(), lr=1e-3, weight_decay=1e-4)

    modele.train()
    for _ in range(EPOQUES):
        ordre = torch.randperm(len(app), device=APPAREIL)
        for debut in range(0, len(app), LOT):
            idx = ordre[debut:debut + LOT]
            lot_volumes = augmenter([v[idx] for v in V_app], axes)
            lot_tab = None if T_app is None else T_app[idx]
            optimiseur.zero_grad()
            sortie = perte(modele(lot_volumes, lot_tab), y_app[idx])
            sortie.backward()
            optimiseur.step()

    modele.eval()
    with torch.no_grad():
        probabilites = torch.sigmoid(modele(V_test, T_test)).cpu().numpy()
    return probabilites

In [ ]:
# --- Les 4 configurations, evaluees sur les memes plis ---
configurations = {
    "CNN — IRM seule": ([IRM], [AXE_GD["IRM"]], None),
    "CNN — PET seule": ([PET], [AXE_GD["PET"]], None),
    "CNN — IRM + PET": ([IRM, PET], [AXE_GD["IRM"], AXE_GD["PET"]], None),
    "CNN — IRM + PET + tabulaire": ([IRM, PET], [AXE_GD["IRM"], AXE_GD["PET"]], TABULAIRE),
}

resultats = {nom: [] for nom in configurations}

for numero, (app, test) in enumerate(CV.split(IRM, y), start=1):
    for nom, (volumes, axes, tab) in configurations.items():
        proba = entrainer_evaluer(volumes, axes, tab, app, test)
        resultats[nom].append(roc_auc_score(y[test], proba))
    print(f"Pli {numero}/5 termine", flush=True)

tableau = pd.DataFrame({
    "ROC-AUC": {nom: f"{np.mean(v):.3f} ± {np.std(v):.3f}" for nom, v in resultats.items()},
})
tableau["_m"] = [np.mean(resultats[n]) for n in tableau.index]
tableau = tableau.sort_values("_m", ascending=False)[["ROC-AUC"]]
tableau.index.name = "Configuration"

print("\n=== CNN 3D — progression, 112 participants ===")
display(tableau)

In [ ]:
# --- Comparaison avec le notebook 10 et enregistrement ---
meilleur_cnn = max(np.mean(v) for v in resultats.values())
print(f"Meilleur CNN : AUC {meilleur_cnn:.3f}")
print("Baseline tabulaire (notebook 08) : AUC 0,803")
print("A comparer aussi avec le tableau RQ1 du notebook 10.")

SORTIE = C.sortie("cnn_3d")
tableau.to_csv(SORTIE / "resultats_CNN.csv", encoding="utf-8-sig")
pd.DataFrame(resultats).to_csv(SORTIE / "auc_par_pli.csv", index=False, encoding="utf-8-sig")
print("\nResultats enregistres dans :", SORTIE)

## Ce qu'il faut livrer a Membre B (point d'arret 2)

- `resultats_CNN.csv` (le tableau ci-dessus) et `auc_par_pli.csv` ;
- une phrase de conclusion, par exemple :
  *« Le meilleur CNN atteint une AUC de X, contre Y pour les caracteristiques
  resumees (notebook 10) et 0,80 pour le modele tabulaire. »*

## Si les resultats sont mauvais (AUC proche de 0,5)

C'est **attendu** avec 112 sujets — ce n'est pas une erreur de ta part. Pistes,
dans l'ordre :
1. reduire encore le reseau (`canaux=(8, 16, 32)`) ;
2. baisser le nombre d'epoques (surapprentissage) ;
3. augmenter le dropout.

Et surtout : **documenter le resultat tel quel**. Montrer qu'un CNN ne suffit pas
sur un petit echantillon est une conclusion scientifique legitime, qui rejoint la
litterature sur les limites du deep learning en petite cohorte.